In [4]:
import pandas as pd

# Fair / Unfair Classification

### Load TSV files
We’ll only need to focus on these two columns for training the classifier.
* text (the sentence)
* label (0 for fair, 1 for unfair)

In [5]:
train_df = pd.read_csv("dataset/claudette_train_merged.tsv", sep='\t')[['text', 'label']]
val_df = pd.read_csv("dataset/claudette_test_merged.tsv", sep='\t')[['text', 'label']]
test_df = pd.read_csv("dataset/claudette_val_merged.tsv", sep='\t')[['text', 'label']]

In [6]:
# Preview a few rows
train_df.head()

,text,label
0,websites & communications terms of use,0
1,please read the terms of this entire document ...,0
2,by accessing or signing up to receive communic...,1
3,our websites include multiple domains such as ...,0
4,you may also recognize our websites by nicknam...,0


### Tokenization

In [7]:
from transformers import BertTokenizer

# Load pre-trained BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [8]:
def tokenize_data(df, tokenizer, max_length=512):
    """Tokenizes the text column of the dataframe"""
    return tokenizer(df['text'].tolist(), 
                     padding=True,  # Pad sequences to the longest one in the batch
                     truncation=True,  # Truncate to max_length
                     max_length=max_length,  # Maximum length of tokens
                     return_tensors='pt')  # Return as PyTorch tensors

# Tokenize train, validation, and test datasets
train_encodings = tokenize_data(train_df, tokenizer, max_length=128)
val_encodings = tokenize_data(val_df, tokenizer, max_length=128)
test_encodings = tokenize_data(test_df, tokenizer, max_length=128)

[#sentences , length_of_the_longest_sentence(others are padded)]

- attention_mask: shows which are padding which are not
- input_ids: token ids for tokens in the sentence. Rest is 0 as padding
- token_type_ids: not-relevant, used for sentence pair differentiating - e.g. input + context

In [9]:
train_encodings.keys(), train_encodings.input_ids.shape, train_encodings.attention_mask.shape, train_df['label'].shape, val_encodings.input_ids.shape, val_encodings.attention_mask.shape, val_df['label'].shape, test_encodings.input_ids.shape, test_encodings.attention_mask.shape, test_df['label'].shape

(dict_keys(['input_ids', 'token_type_ids', 'attention_mask']),
 torch.Size([8354, 128]),
 torch.Size([8354, 128]),
 (8354,),
 torch.Size([3784, 128]),
 torch.Size([3784, 128]),
 (3784,),
 torch.Size([8279, 128]),
 torch.Size([8279, 128]),
 (8279,))

### Prepare Dataset for PyTorch

In [10]:
import torch
from torch.utils.data import Dataset

class ClaudetteDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Return a dictionary with input_ids, attention_mask, token_type_ids, and label
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

In [11]:
train_dataset = ClaudetteDataset(train_encodings, train_df['label'].tolist())
val_dataset = ClaudetteDataset(val_encodings, val_df['label'].tolist())
test_dataset = ClaudetteDataset(test_encodings, test_df['label'].tolist())

### Load BERT model for Sequence Classification

This does several things:
- Loads a pretrained BERT base model (bert-base-uncased)
- Adds a classification layer on top (with 2 output logits)
- Initializes the classification layer with random weights
- Keeps the BERT weights as-is (unless you fine-tune it, which we will)

In [12]:
from transformers import BertForSequenceClassification

# Load BERT with a classification head (2 labels: fair and unfair)
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
import torch

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
device

device(type='cpu')

### Training

Dataset is highly imbalanced (10% unfair, 90% fair), so we need to compensate for the class imbalance during training. We’ll do this using weighted loss, so that the model pays more attention to the minority class (1: unfair) during training.

1. Class Weights: You calculate weights for the minority class (unfair) so the model will pay more attention to it.
2. Custom Loss Function: You modify the default loss function to use the class weights during training.
3. Training Arguments: You define how the model will be trained, including batch size, learning rate, and number of epochs.
4. Metrics: You define accuracy and F1 score to evaluate the model during training.
5. Trainer: You create the Trainer object, which will handle the training process.

In [14]:
from sklearn.utils.class_weight import compute_class_weight
import torch
import numpy as np

# Compute weights for classes 0 (fair) and 1 (unfair)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=train_df['label'].values
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

print("Class weights:", class_weights)


Class weights: tensor([0.5586, 4.7683])


Since Trainer uses cross-entropy internally, we’ll override the loss function using compute_loss:

In [15]:
from transformers import Trainer
import torch.nn as nn

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fn = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [16]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",        # evaluate each epoch
    save_strategy="epoch",        # save checkpoint each epoch
    logging_strategy="epoch",     # log metrics each epoch
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

In [17]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
    }

In [18]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

This will:
- Train with weighted loss
- Evaluate on val set after each epoch
- Track accuracy and F1
- Save the best model

In [19]:
# USE GOOGLE COLAB TO TRAIN
# Uncomment the following line to start training
# trainer.train()

In [20]:
from transformers import BertForSequenceClassification

# Load the model from the output directory
best_model_path = "./results_ck"  # Replace with the path to the checkpoint you want (e.g., checkpoint-XXX)
model = BertForSequenceClassification.from_pretrained(best_model_path)

In [21]:
# Assuming the test dataset has been prepared as 'test_dataset'
test_results = trainer.evaluate(test_dataset)

# Print test results (loss, accuracy, F1-score, etc.)
print(test_results)


{'eval_loss': 0.7107157707214355, 'eval_model_preparation_time': 0.0054, 'eval_accuracy': 0.8838023915931876, 'eval_f1': 0.002074688796680498, 'eval_runtime': 877.9313, 'eval_samples_per_second': 9.43, 'eval_steps_per_second': 0.148}


In [23]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Get predictions from the model
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

# Confusion matrix
cm = confusion_matrix(labels, preds)

# Plot confusion matrix
plt.figure(figsize=(6, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Fair', 'Unfair'], yticklabels=['Fair', 'Unfair'])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import roc_curve, auc

# Compute ROC curve
fpr, tpr, thresholds = roc_curve(labels, predictions.predictions[:, 1])  # probabilities for class 1 (unfair)
roc_auc = auc(fpr, tpr)

# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='red', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.show()


In [ ]:
from sklearn.metrics import precision_recall_curve

# Compute precision-recall curve
precision, recall, _ = precision_recall_curve(labels, predictions.predictions[:, 1])
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='blue')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.show()


In [ ]:
from sklearn.metrics import classification_report

# Classification report (precision, recall, F1, etc.)
class_report = classification_report(labels, preds)
with open("classification_report.txt", "w") as f:
    f.write(class_report)

# Save the confusion matrix plot
plt.figure(figsize=(6, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Fair', 'Unfair'], yticklabels=['Fair', 'Unfair'])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.savefig('confusion_matrix.png')
